# Update the Hugging Face cards

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chingkheinganba231005/meitei-mayek-word-recognition/blob/main/notebooks/update_hf_cards.ipynb)

Rewrites the texts of the Hugging Face model repository and Space
(`Chingkheinganba/handwritten-meitei-mayek-word-recognition`) from the results in this
repository, and uploads them:

- the model card (`README.md`) and `results.json` of the model repository;
- the Space card (`README.md`) and the paragraph under the demo page (`about` in `config.json`).

The network, the language model and the demo page itself are not touched. No GPU and no
Google Drive are needed.

**Before running:** add a Hugging Face token with write access as a Colab secret called
`HF_TOKEN` (key icon in the left bar; switch on notebook access). Never paste the token into a
cell.

Run the cells in order. Cell 2 shows the new texts; cell 3 uploads them once `UPLOAD = True`;
cell 4 reads back what Hugging Face now serves.

In [ ]:
# 1. Settings, and the repository (branch main)
REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
BRANCH = "main"
HF_ID = "Chingkheinganba/handwritten-meitei-mayek-word-recognition"   # model repository and Space
UPLOAD = False   # True to upload the new texts

import json, os, shutil, subprocess, sys
from pathlib import Path

REPO_DIR, OUT = "/content/repo", "/content/hf_cards"
shutil.rmtree(REPO_DIR, ignore_errors=True)
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR],
                   capture_output=True, text=True)
if r.returncode:
    raise SystemExit("git clone failed: " + r.stderr)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
print("repository at", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Write the new texts from the results files, and show them
shutil.rmtree(OUT, ignore_errors=True)
r = subprocess.run([sys.executable, "scripts/update_cards.py", "--results", "results", "--out", OUT, "--hf-id", HF_ID],
                   capture_output=True, text=True)
print(r.stdout + r.stderr)
if r.returncode:
    raise SystemExit("the cards could not be written")
print("--- paragraph under the demo page ---")
print(Path(OUT, "space/about.txt").read_text(encoding="utf-8"))
print("\n--- model card ---")
print(Path(OUT, "model/README.md").read_text(encoding="utf-8"))

In [ ]:
# 3. Upload (only with UPLOAD = True in cell 1)
if UPLOAD:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    r = subprocess.run([sys.executable, "scripts/update_cards.py", "--results", "results", "--out", OUT,
                        "--hf-id", HF_ID, "--upload"], capture_output=True, text=True, env=dict(os.environ, HF_TOKEN=token))
    print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit("upload failed: is the HF_TOKEN secret set, with write access, and allowed for this notebook?")
else:
    print("UPLOAD is False: nothing was uploaded. Set UPLOAD = True in cell 1 and run cells 1-4 again.")

In [ ]:
# 4. Read back what Hugging Face now serves, and compare with what was uploaded
if UPLOAD:
    from huggingface_hub import hf_hub_download
    fetch = lambda kind, name: Path(hf_hub_download(HF_ID, name, repo_type=kind, token=token,
                                                    force_download=True)).read_text(encoding="utf-8")
    checks = {
        "model card": fetch("model", "README.md") == Path(OUT, "model/README.md").read_text(encoding="utf-8"),
        "model results.json": json.loads(fetch("model", "results.json"))
                              == json.loads(Path(OUT, "model/results.json").read_text(encoding="utf-8")),
        "Space card": fetch("space", "README.md") == Path(OUT, "space/README.md").read_text(encoding="utf-8"),
        "demo page paragraph": json.loads(fetch("space", "config.json"))["about"]
                               == Path(OUT, "space/about.txt").read_text(encoding="utf-8"),
    }
    for what, same in checks.items():
        print(f"{what}: {'updated' if same else 'DIFFERENT from the new text'}")
    print(f"\nModel: https://huggingface.co/{HF_ID}\nDemo:  https://huggingface.co/spaces/{HF_ID}")
else:
    print("UPLOAD is False: nothing to check.")